# Analysing a NEAR account

Sync one testnet account into a temporary database, then look at its activity with pandas.

In [ ]:
import tempfile
from pathlib import Path

from multichain_indexer import Indexer

ACCOUNT = "f781edbfe07287152e3912cb698af4155234df3c2d9c23bf625f675b8f6de5fa"
folder = tempfile.TemporaryDirectory()
indexer = Indexer("near", database=Path(folder.name) / "example.db")
indexer.sync(ACCOUNT)

Exports carry the shared fields as columns and the `native` block as JSON. `amount` and `fee` are text, because a yoctoNEAR amount does not fit in 64 bits. Gas is in gas units: 1 Tgas is 10^12.

In [ ]:
import io
import json

import pandas as pd

frame = pd.read_csv(io.StringIO(indexer.export("csv", account=ACCOUNT)), dtype={"fee": str})
native = frame["native"].map(json.loads)
frame["kind"] = native.map(lambda n: n["kind"])
frame["tgas"] = native.map(lambda n: n["gas_burnt"] / 10**12)
frame["receipts"] = native.map(lambda n: len(n["receipts"]))
frame[["time", "kind", "status", "tgas", "receipts"]].head()

In [ ]:
frame.groupby("kind")[["tgas", "receipts"]].agg(["count", "sum", "max"])

In [ ]:
indexer.metrics(days=365, account=ACCOUNT).native

In [ ]:
indexer.close()
folder.cleanup()